In [1]:
!pip -q install llama-index

In [38]:
import sys
from pathlib import Path

project_root = Path('/work')
sys.path.append(str(project_root))

import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForMaskedLM
from scipy.sparse import csr_matrix, load_npz
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

import numpy as np

import re
import pickle
import unicodedata
import logging 
import random
import time
import json
from typing import List, Union, Dict
from src.utils.data_utils import load_documents, select_random_documents
from contextlib import contextmanager
from llama_index.core import Document


# Import the evaluation functions
from src.utils.evaluation import (
    perform_analysis,
    perform_single_match_analysis,
    print_single_match_analysis,
    print_sample_results,
    calculate_prefix_success_at_k,
    perform_prefix_match_analysis,
    print_prefix_match_analysis
)

# ---------------------------
# Timing context manager
# ---------------------------
@contextmanager
def timer(name):
    omitted_names = {
        "Selecting a random document for sanity check",
        "Encoding query",
        "Searching in Milvus",
        "Processing search results"
    }
    start = time.time()
    yield
    end = time.time()
    if name not in omitted_names and not name.startswith("Retrieval for document"):
        print(f"[Timing] {name}: {end - start:.4f} seconds")

# ---------------------------
# SpladeSearcher class
# ---------------------------

class SpladeSparseVectorizer:
    """
    SPLADE (SParse Lexical AnD Expansion) vectorizer for Spanish documents.
    Uses a pre-trained Spanish BERT model to create sparse document representations.
    """
    
    def __init__(self, model_name="dccuchile/bert-base-spanish-wwm-cased", 
                 max_length=512, batch_size=1, device=None):
        """
        Initialize SPLADE vectorizer with conservative defaults for memory.
        
        Args:
            model_name: HuggingFace model name for Spanish BERT
            max_length: Maximum sequence length for tokenization (reduced default)
            batch_size: Batch size for processing documents (reduced default)
            device: Device to run the model on (auto-detected if None)
        """
        self.model_name = model_name
        self.max_length = max_length
        self.batch_size = batch_size
        
        # Set device
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)
        
        print(f"Using device: {self.device}")
        
        # Load tokenizer and model with memory optimizations
        print(f"Loading tokenizer and model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Load model with consistent precision
        if self.device.type == 'cuda':
            self.model = AutoModelForMaskedLM.from_pretrained(
                model_name, 
                torch_dtype=torch.float32  # Use float32 for consistency
            )
        else:
            self.model = AutoModelForMaskedLM.from_pretrained(model_name)
            
        self.model.to(self.device)
        self.model.eval()
        
        # Vocabulary size
        self.vocab_size = len(self.tokenizer.vocab)
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Model loaded with batch_size={batch_size}, max_length={max_length}")
        
        # Document statistics (will be computed during fit)
        self.fitted = False
        
    def _extract_splade_features(self, texts: List[str]) -> csr_matrix:
        """
        Extract SPLADE sparse features from texts with memory optimization.
        
        Args:
            texts: List of text strings
            
        Returns:
            Sparse matrix of SPLADE features
        """
        # Use lists to store sparse matrix components efficiently
        rows, cols, data = [], [], []
        current_row = 0
        
        print(f"Processing {len(texts)} texts in batches of {self.batch_size}...")
        batch_start_time = time.time()
        
        # Process in batches with progress tracking
        for batch_idx in range(0, len(texts), self.batch_size):
            batch_texts = texts[batch_idx:batch_idx + self.batch_size]
            
            # Enhanced progress tracking for large datasets
            current_batch = batch_idx // self.batch_size + 1
            total_batches = (len(texts) - 1) // self.batch_size + 1
            
            if current_batch % max(1, total_batches // 20) == 0 or current_batch == 1:  # Show progress every 5%
                progress = (current_batch / total_batches) * 100
                print(f"Progress: {progress:.1f}% - Batch {current_batch}/{total_batches}")
                
                # Estimate remaining time
                if current_batch > 1:
                    elapsed = time.time() - batch_start_time
                    avg_time_per_batch = elapsed / (current_batch - 1)
                    remaining_batches = total_batches - current_batch
                    est_remaining = avg_time_per_batch * remaining_batches
                    minutes, seconds = divmod(est_remaining, 60)
                    print(f"Estimated time remaining: {int(minutes)}m {int(seconds)}s")
            
            try:
                # Tokenize batch
                inputs = self.tokenizer(
                    batch_texts,
                    max_length=self.max_length,
                    padding=True,
                    truncation=True,
                    return_tensors="pt"
                ).to(self.device)
                
                with torch.no_grad():
                    # Get model outputs
                    outputs = self.model(**inputs)
                    logits = outputs.logits  # [batch_size, seq_len, vocab_size]
                    
                    # Apply log(1 + ReLU) activation to get sparse weights
                    sparse_weights = torch.log1p(torch.relu(logits))
                    
                    # Max pooling over sequence dimension to get document representation
                    # Shape: [batch_size, vocab_size]
                    doc_sparse = torch.max(sparse_weights, dim=1)[0]
                    
                    # Process each document in the batch
                    for batch_doc_idx, doc_weights in enumerate(doc_sparse):
                        # Apply threshold to reduce sparsity and memory usage
                        threshold = 0.1  # Only keep values above this threshold
                        significant_mask = doc_weights > threshold
                        
                        if significant_mask.any():
                            # Get indices and values for significant weights
                            indices = torch.nonzero(significant_mask).squeeze(-1)
                            values = doc_weights[indices]
                            
                            # Convert to CPU and add to sparse matrix components
                            indices_cpu = indices.cpu().numpy().astype(np.int32)
                            values_cpu = values.cpu().numpy().astype(np.float32)
                            
                            # Add to sparse matrix components
                            rows.extend([current_row] * len(indices_cpu))
                            cols.extend(indices_cpu.tolist())
                            data.extend(values_cpu.tolist())
                        
                        current_row += 1
                
                # Clear GPU memory after each batch
                del inputs, outputs, logits, sparse_weights, doc_sparse
                torch.cuda.empty_cache() if torch.cuda.is_available() else None
                
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"GPU memory error at batch {batch_idx // self.batch_size + 1}. Trying to continue...")
                    torch.cuda.empty_cache() if torch.cuda.is_available() else None
                    # Try processing with smaller batch size
                    return self._extract_splade_features_fallback(texts)
                else:
                    raise e
        
        # Create sparse matrix from components with explicit dtype
        if rows:
            # Ensure all components are the right type
            rows = np.array(rows, dtype=np.int32)
            cols = np.array(cols, dtype=np.int32)
            data = np.array(data, dtype=np.float32)
            
            sparse_matrix = csr_matrix((data, (rows, cols)), 
                                     shape=(len(texts), self.vocab_size),
                                     dtype=np.float32)
        else:
            # Handle edge case of no features
            sparse_matrix = csr_matrix((len(texts), self.vocab_size), dtype=np.float32)
        
        print(f"Created sparse matrix with shape: {sparse_matrix.shape}")
        print(f"Non-zero elements: {sparse_matrix.nnz}")
        
        return sparse_matrix
    
    def _extract_splade_features_fallback(self, texts: List[str]) -> csr_matrix:
        """
        Fallback method with smaller batch size for memory issues.
        """
        print("Using fallback method with batch size 1...")
        original_batch_size = self.batch_size
        self.batch_size = 1
        
        try:
            result = self._extract_splade_features(texts)
            return result
        finally:
            self.batch_size = original_batch_size
    
    def fit(self, documents: List[Union[Document, str]]):
        """
        Fit the SPLADE vectorizer to documents.
        
        Args:
            documents: List of Document objects or strings
        """
        print("Extracting text from documents...")
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        
        print(f"Fitting SPLADE vectorizer on {len(texts)} documents...")
        
        # For SPLADE, we don't need to compute global statistics like BM25
        # The model learns the sparse representations end-to-end
        self.fitted = True
        print("SPLADE vectorizer fitted successfully.")
        
        return self
    
    def transform(self, documents: List[Union[Document, str]],
                 batch_size: int | None = None) -> csr_matrix:
        """
        Transform documents to SPLADE sparse representations.
        
        Args:
            documents: List of Document objects or strings
            
        Returns:
            Sparse matrix of SPLADE features
        """
        if not self.fitted:
            raise ValueError("Vectorizer must be fitted before transforming documents.")
        
        if isinstance(documents, str):
            documents = [documents]
        texts = [d.text if isinstance(d, Document) else d for d in documents]
        
        if batch_size is not None:
            old_bs = self.batch_size
            self.batch_size = batch_size
            
        try:
            sparse_matrix = self._extract_splade_features(texts)
        finally:
            if batch_size is not None:
                self.batch_size = old_bs # restore batch size
        
        return sparse_matrix

    def load_index(self, index_path: Path):
        """
        Load embeddings and documents for retrieval.
        """
        self.embeddings = load_npz(index_path / "embeddings.npz")
        with open(index_path / "documents.pkl", "rb") as f:
            self.documents = pickle.load(f)
        
        # L2-normalize the sparse embeddings (row-wise)
        self.embeddings = normalize(self.embeddings, norm='l2', axis=1)
        print(f"Loaded {self.embeddings.shape[0]} normalized documents.")

    def search(self, query: str, top_k: int = 10) -> List[Dict[str, Union[str, float]]]:
        """
        Search the index using a query.
    
        Returns:
            List of dicts with 'item_key', 'text', and 'score'
        """
        if not hasattr(self, "embeddings"):
            raise ValueError("Index not loaded. Call load_index(path) first.")
        
        print(f"Encoding and normalizing query: {query}")
        query_vec = self.transform([query])
        query_vec = normalize(query_vec, norm='l2', axis=1)
    
        scores = query_vec.dot(self.embeddings.T).toarray().flatten()
        top_indices = scores.argsort()[::-1][:top_k]
    
        results = []
        for i in top_indices:
            doc = self.documents[i]
            item_key = doc.metadata.get("item_key", f"doc_{i}")
            text = doc.get('text') if isinstance(doc, dict) else getattr(doc, 'text', str(doc))
            results.append({
                "item_key": item_key,
                "text": text,
                "score": float(scores[i])
            })
        return results

    def search_by_item_key(self, item_key: str):
        """
        Retrieve a document by item_key, including its SPLADE vector.
    
        Returns:
            dict with keys: 'item_key', 'text', 'vector' (a sparse row vector), or None if not found
        """
        if not hasattr(self, "documents") or not hasattr(self, "embeddings"):
            raise ValueError("Index not loaded. Call load_index() first.")
        
        for i, doc in enumerate(self.documents):
            # Extract key
            key = doc.metadata.get("item_key") if isinstance(doc, dict) else getattr(doc.metadata, "item_key", None)
            text = doc.get("text") if isinstance(doc, dict) else getattr(doc, "text", str(doc))
    
            if key == item_key:
                vector = self.embeddings[i]  # sparse row
                return {
                    "item_key": key,
                    "text": text,
                    "vector": vector
                }
    
        return None

    @classmethod
    def from_saved(cls, index_path: Path) -> "SpladeSparseVectorizer":
        """
        Restore a saved vectorizer instance from disk.
        
        Args:
            index_path: Path where vectorizer.pkl is stored
        
        Returns:
            An instance of SpladeSparseVectorizer with config restored
        """
        with open(index_path / "vectorizer.pkl", "rb") as f:
            state = pickle.load(f)
        
        instance = cls(
            model_name=state['model_name'],
            max_length=state['max_length'],
            batch_size=state['batch_size']
        )
        instance.fitted = state['fitted']
        return instance

# ---------------------------
# Main pipeline function
# ---------------------------

# Fixed version of the retrieval loop section
def main(file: str, *, sample_size: int = 16_590, batch_q: int = 32):
    """End‑to‑end retrieval quality analysis."""
    overall_start = time.time()

    # ------------------------------------------------------------------
    # 1) Initialise vectoriser & ANN index **once** 
    # ------------------------------------------------------------------
    index_path = Path("/work/sandbox/splade/index/bert-base-spanish-wwm-cased_texto_sparse_t0.5_k200")
    vectorizer = SpladeSparseVectorizer.from_saved(index_path)
    vectorizer.load_index(index_path)

    # ------------------------------------------------------------------
    # 2) Load *texto* corpus (for sanity check + item‑key lookup)
    # ------------------------------------------------------------------
    documents_path_texto = Path(f"/work/data/processed/{file}_texto.pkl")
    with timer("Loading texto documents"):
        documents_texto: List = load_documents(documents_path_texto)
        if documents_texto is None:
            print("Failed to load texto documents. Exiting.")
            return

    # ---------------------------
    # Sanity Check – single query
    # ---------------------------
    with timer("Sanity check"):
        random_doc = random.choice(documents_texto)
        random_doc_key = random_doc.metadata.get("item_key")
        random_doc_text = random_doc.text
        print(f"Item key: {random_doc_key}\nText: {random_doc_text}\n")

        sanity_results = vectorizer.search(random_doc_text, top_k=5)

    def get_item_key(rec: dict) -> str:
        return (
            rec.get("metadata", {}).get("item_key")  # preferred
            or rec.get("item_key")                   # fallback
            or "N/A"
        )
    
    print("Sanity Retrieval Results:")
    for i, res in enumerate(sanity_results, 1):
        print(f"\nResult {i}:")
        print(f"Item key: {get_item_key(res)}")
        print(f"Text: {res['text']}")
        print(f"Score: {res['score']:.4f}")

    # ------------------------------------------------------------------
    # 3) Load & sample *resumen* docs
    # ------------------------------------------------------------------
    documents_path_resumen = Path(f"/work/data/processed/{file}_resumen.pkl")
    with timer("Loading resumen documents"):
        documents_resumen: List[Tuple[str, str]] = load_documents(documents_path_resumen)
        if documents_resumen is None:
            print("Failed to load resumen documents. Exiting.")
            return

    # sample with 99 % confidence interval cap
    sample = min(len(documents_resumen), sample_size)
    with timer("Selecting random resumen documents"):
        random_resumen = select_random_documents(documents_resumen, sample)

    # ---------------------------------------------------------------
    # 4) ULTRA-OPTIMIZED: Encode once, search multiple times
    # ---------------------------------------------------------------
    with timer("Performing ANN retrieval for all queries"):
        all_results = []

        # Build a dict once for O(1) item_key lookup
        key_to_idx = {}
        for i, doc in enumerate(vectorizer.documents):
            if hasattr(doc, 'metadata') and isinstance(doc.metadata, dict):
                key = doc.metadata.get("item_key")
            elif isinstance(doc, dict):
                key = doc.get("item_key")
            else:
                key = f"doc_{i}"
            
            if key:
                key_to_idx[key] = i

        total = len(random_resumen)
        print(f"Processing {total} resumen documents with batch encoding")
        
        # Process in batches to balance memory and efficiency
        BATCH_SIZE = 64  # Reasonable batch size
        
        for batch_start in range(0, total, BATCH_SIZE):
            batch_end = min(batch_start + BATCH_SIZE, total)
            batch = random_resumen[batch_start:batch_end]
            
            elapsed = time.time() - overall_start
            progress = batch_start / total * 100
            rate = batch_start / elapsed if elapsed > 0 else 0
            eta = (total - batch_start) / rate if rate > 0 else 0
            
            print(f"Batch {batch_start//BATCH_SIZE + 1}/{(total-1)//BATCH_SIZE + 1}: "
                  f"{batch_start}-{batch_end-1} ({progress:.1f}%) - "
                  f"Rate: {rate:.1f} docs/sec - ETA: {eta/60:.1f}min")
            
            # Extract texts for this batch
            texts = [text for _, text in batch]
            
            try:
                # Encode the entire batch ONCE
                print(f"  Encoding {len(texts)} queries...")
                old_bs = vectorizer.batch_size
                vectorizer.batch_size = min(batch_q, len(texts))
                
                batch_vecs = vectorizer.transform(texts)
                batch_vecs = normalize(batch_vecs, norm="l2", axis=1)
                
                vectorizer.batch_size = old_bs
                torch.cuda.empty_cache()
                
                print(f"  Searching {len(texts)} queries...")
                # Use scipy's efficient sparse operations
                # This is the fastest way to compute similarities
                for i, (item_key, query_text) in enumerate(batch):
                    # Get single query vector as a row matrix
                    query_vec = batch_vecs[i:i+1]  # Keep as 1x31002 matrix
                    
                    # Fast sparse matrix multiplication
                    scores = query_vec.dot(vectorizer.embeddings.T).toarray().flatten()
                    
                    # Get top-k indices efficiently
                    top_k = 10
                    if len(scores) > top_k:
                        top_indices = np.argpartition(scores, -top_k)[-top_k:]
                        top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
                    else:
                        top_indices = np.argsort(scores)[::-1]
                    
                    # Format ANN results
                    ann_hits = []
                    for idx in top_indices:
                        doc = vectorizer.documents[idx]
                        
                        # Extract item_key and text from Document object
                        doc_key = doc.metadata.get("item_key", f"doc_{idx}")
                        doc_text = doc.text

                        ann_hits.append({
                            "item_key": doc_key,
                            "text": doc_text,
                            "score": float(scores[idx])
                        })

                    # Get target document score
                    target_idx = key_to_idx.get(item_key)
                    if target_idx is not None:
                        target_doc = vectorizer.documents[target_idx]
                        if isinstance(target_doc, dict):
                            target_text = target_doc.get("text", "")
                        else:
                            target_text = getattr(target_doc, "text", str(target_doc))
                        
                        target_score = float(scores[target_idx])
                    else:
                        target_score = 0.0
                        target_text = ""
                        if batch_start == 0 and i < 5:  # Only warn for first few
                            print(f"    Warning: Target not found for item_key: {item_key}")
                    
                    # Store results
                    all_results.append({
                        "resumen_item_key": item_key,
                        "resumen_text": query_text,
                        "target_text": target_text,
                        "target_score": target_score,
                        "texto_results": ann_hits,
                    })
                    
                    # Progress within batch
                    if (i + 1) % 16 == 0:
                        print(f"    Processed {i+1}/{len(batch)} queries in batch")
                
            except Exception as e:
                print(f"  Error processing batch: {e}")
                continue
            
            # Clear memory every few batches
            if batch_start % (BATCH_SIZE * 4) == 0:
                torch.cuda.empty_cache()
        
        print(f"Completed processing {len(all_results)} documents")

    # ---------------------------------------------------------------
    # 5) Evaluations 
    # ---------------------------------------------------------------
    print(f"\nCompleted retrieval for {len(all_results)} documents")
    
    with timer("Performing analysis on retrieval results"):
        analysis_results = perform_analysis(all_results)

    print("\nAnalysis Results:")
    for metric, value in analysis_results.items():
        print(f"{metric}: {value:.4f}")

    with timer("Performing single-match analysis"):
        sm_metrics = perform_single_match_analysis(all_results, top_k=10)
        print_single_match_analysis(sm_metrics, top_k=10)

    with timer("Performing prefix-match analysis"):
        prefix_len = 6
        pm_metrics = perform_prefix_match_analysis(all_results, top_k=10, prefix_length=prefix_len)
        print_prefix_match_analysis(pm_metrics, top_k=10, prefix_length=prefix_len)

    with timer("Printing sample results"):
        print_sample_results(all_results, num_samples=5, top_k=5)

    print(f"\nTotal execution time: {time.time() - overall_start:.2f} seconds")

In [39]:
main('OEB')

Using device: cuda
Loading tokenizer and model: dccuchile/bert-base-spanish-wwm-cased
Vocabulary size: 31002
Model loaded with batch_size=2, max_length=512
Loaded 47514 normalized documents.
[Timing] Loading texto documents: 3.7593 seconds
Item key: OEB020dcecb
Text: Canalización hormigonada de  8  tubos de PVC de 110 mm de diámetro en terreno rocoso, incluso la demolición de roca dura, el relleno y el compactado de la zanja,  el suministro y el montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo.  Trabajo: Cualquier franja horaria Banda de mantenimiento: i < 3 horas Condiciones de ejecución: Volumen escaso

Encoding and normalizing query: Canalización hormigonada de  8  tubos de PVC de 110 mm de diámetro en terreno rocoso, incluso la demolición de roca dura, el relleno y el compactado de la zanja,  el suministro y el montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los con